### **ACID - PART 4a - Compute background function**

# --- --- ---

### This notebook computes and saves a background function. Part 4b notebook imports the background function and uses it for illumination correction.

### Skeep this notebook if a background function has already been computed and saved.

# --- --- ---

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2026/02/06


## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [1]:
# Import required modules
import datetime
import os
from pathlib import Path
import numpy as np
import pandas as pd
import tifffile
import napari
from utils.listdirNHF import listdirNHF
from utils.get_defaults import default_file_name
from utils.fov_axis_utils import get_fov_ch_shape
# from utils.mksubdir import mk_subdir
# from image_processing.extract_metadata import extract_bioio_scene_metadata
# from image_processing.name_metadata import extract_name_metadata
# from image_processing.make_imagej_metadata import imagej_compatible_metadata_dict
# from utils.open_image import bioio_open_image
# from utils.save_image import tifffile_save_ometiff
# from image_processing.save_metadata import save_xml_string



### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [ ]:
# indicate the path to the directory storing the metadata file - NOTE: this is expected to be the metadata_df saved
# from part1 notebook
# metadata_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\proc_metadata"
metadata_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\proc_metadata"

# indicate the path to the directory storing the fields of view
# fov_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\fov"
fov_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\fov"

# # indicate the path to the directory where outputs will be saved
# NOTE: it the directory does not exist, the pipeline will try to create it
# output_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\background"
output_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\develop\260112_background_funct"


# name of the metadata file - NOTE: this is expected to be the metadata_df saved
# from part3 notebook
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "", aka it is not "None")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved csv file will be used.
# By default, the timestamp of the last file modification is used. As an alternative it is possible to use the
# date saved in the name (change below parameter metadata_from_file_name to True -
# see utils.get_defaults.default_file_name documentation).
metadata_file_name = "default"


# ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"
# # --- parameters to select the train set ---
# indicate the name of the column indicating whether the row belongs to train or test set
is_train_column="is_train"

# indicate the value signalling that a row (aka a field of view) belongs to the train set
train_val=1

# # --- parameters to select the non flagged fields of view ---
# indicate the name of the column indicating whether the row belongs is or is not flagged
flag_column="flag"

# indicate the value signalling that a row (aka a field of view) is flagged
# NOTE: any value which is not this value will be considered as unflagged - only unflagged fields of view
# will be used in the background function computation
flag_value=1


# # --- parameters used for importing the default metadata dataframe ---
# Indicate the part of the file name to use to select files into metadata_directory to be used for selecting the default
# file
default_metadata_file_target = ".csv" # files with this string in their name will be selected for the default file selection - if None, all files will be selected
default_metadata_file_exclude = None # files with this string in their name will be excluded from the default file selection - if None, no files will be excluded

# Indicate whether to extract the date information from the file name when selecting the default file
# If False, the date will be extracted from the file's last modified timestamp, if True, from the file name.
# If from_file_name is True, the parameters *_default_separator and *_default_date_position
# should be used to indicate, respectively the separator to use for splitting the file name into tokens and
# the position of the token containing the date information. In addition, the date format used to include the date in the file name
# should be indicated using the parameters *_default_date_format.
# For example, if metadata_from_file_name is True, and the file name is "251126_plate_layout.csv",
# "_" is used as separator, 0 as date position, and '%Y%m%d' as date format.
# Ref to utils.get_defaults.default_file_name for more details.
metadata_from_file_name = False

# separator - used to split the file name and extract the information token with the date
# if metadata_from_file_name is True
metadata_default_separator = '_'

# date position - the position of the date information token after splitting the file name using file_name_separator (above)
# used if metadata_from_file_name is True
metadata_default_date_position = 0

# date format - the format used for including the date in the file name to be opened by default
# used if metadata_from_file_name is True
metadata_default_date_format='%Y%m%d'

# reverse - if True, the file with the most recent date will be returned by default - if False, the opposite
metadata_default_reverse = True


# --- parameters for background function calculation ---
# background function computation method - this is the method to use for calculating the
# background function. Possible options are:
# "median" - the background function will be calculated as the median of the pixel values across the fields of view (after applying the flag and train set filters)
# "mean" - the background function will be calculated as the mean of the pixel values across the fields of view (after applying the flag and train set filters)
background_function_method = "median"

# polynomial degree to use for fitting the polynomial surface after calculating the background
# function using the method indicated above (median or mean).
# if None, no polynomial fitting will be applied and the background function will be the one
# calculated using the method indicated above.
# Recommended to use a degree between 2 and 5 - if the degree is too high, the fitted
# surface may overfit the background function and not generalize well to other fields of view
# NOTE: a background function is calculated for each channel. The same polynomial degree is applied to all channels.
polynomial_degree = 3 # recommended to use a degree 3

# channel axis to be passed to background calculating functions in order to calculate background functions per channel
# this is the axis along which the channels are organized in the field of view arrays.
# For example, if the field of view arrays have shape (channels, height, width), the channel axis is 0.
channel_axis = 0

# field of view column name in the metadata dataframe
fov_column_name = "ome_tif_file_name"

# well column name in the metadata dataframe
well_column_name = "well"

# plate column name in the metadata dataframe
plate_column_name = "experiment"

# within well position column name in the metadata dataframe
# this is used to identify the position of the field of view within the well
# 49 fields of view were acquired per each well, in a 7x7 grid
position_within_well_column_name = "scene"

# null value to be used when a result can't be computed
null_value = np.nan


# --- parameters for saving metadata within the background function image ---
# background function image - name of processing date in metadata - this is the name
# of the entry in the metadata dictionary to save within the background function image.
# The entry indicates the date when the background function was calculated
background_img_meta_date_name = "background_funct_date_yymmdd"

# background function image - project name in metadata - this is the name of the entry
# in the metadata dictionary to save within the background function image. The entry indicates the
# name of the project (e.g. ACID)
background_img_meta_project_name = "project_name"

# background function image - method name in metadata - this is the name of the entry
# in the metadata dictionary to save within the background function image. The entry indicates the
# method used for calculating the background function (e.g. median or mean)
background_img_meta_method_name = "background_funct_method"

# background function image - polynomial degree name in metadata - this is the name of the entry
# in the metadata dictionary to save within the background function image. The entry indicates the
# degree of the polynomial used for fitting the background function (if polynomial fitting is applied)
background_img_meta_poly_degree_name = "background_funct_poly_degree"




# --- parameters for metadata dataframe updating ---
# background function metadata dataframe - method column name - this is the name of the column to be
# added to the metadata dataframe to indicate the method used for calculating the background function
background_df_method_clm_name = "background_funct_method"

# background function metadata dataframe - polynomial degree column name - this is the name of the column to be added to
# the metadata dataframe to indicate the degree of the polynomial used for fitting the background function
background_df_poly_degree_clm_name = "background_funct_poly_degree"

# background function metadata dataframe - computation date column name - this is the name of the column to be added
# to the metadata dataframe to indicate the day when the background function was calculated
background_df_date_clm_name = "background_funct_date"

# background function metadata dataframe - computation date format - this is the format to be used for indicating the
# date when the background function was calculated. This is used for saving the date in the metadata dataframe
background_df_meta_date_format = '%y%m%d'


# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = '_'

# project
project_name = "ACID"

# include indexes when saving pandas dataframes as csv files
save_csv_index = False # if False, the index will not be saved as a separate column in the csv file

# background function image name - date format - this is the format to use for
# indicating the date when the background function was calculated in the background function image name
background_img_name_date_format = '%Y%m%d'

# background function image name - savingword - this is the word to use in the background
# function image name to indicate that the file is a background function image
background_img_savingword = 'background'

# background function image name - file suffix - this is the suffix to use for the
# background function image file name
background_img_file_suffix = ".ome.tif"

# processing metadata dataframe name - savingword
metadata_savingword = "metadata"

# processing metadata dataframe name - file suffix
metadata_file_suffix = f"part{save_file_name_separator}4a.csv"

# processing metadata dataframe name - date format
metadata_date_format = '%Y%m%d'

# hyperparameters dataframe name - date format
hyperparameters_date_format = '%Y%m%d-%H%M%S'

# hyperparameters dataframe name - savingword
hyperparameters_savingword = "hyperparameters"

# hyperparameters dataframe name - file suffix
hyperparameters_file_suffix = f"part{save_file_name_separator}4a.csv"


# --- parameters for saving secondary information ---
# indicate the name of the directory for saving secondary outputs -
# this directory is used to store the hyperparameters used per each run of the pipeline
secondary_output_directory = "secondary_output"
exist_ok = True # if the secondary output directory already exists, do not raise an error



### Create output directory and secondary output directory if they don't exist

##### Output directory stores the computed background function
##### Secondary output directory is used to store the hyperparameters used per each run of the pipeline

Run the following cell.

Don't modify the following cell.

In [3]:
# create the output_directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=exist_ok)

# create the path to secondary_output directory
secondary_output_path = os.path.join(os.getcwd(), secondary_output_directory)

# create the secondary_output directory if it doesn't exist
if not os.path.exists(secondary_output_path):
    os.makedirs(secondary_output_path, exist_ok=exist_ok)


#### Open the metadata dataframe - this is expected to be the output of part 3

Run the following cell.

Don't modify the following cell.

In [4]:

# check if using the default metadata data frame (the most recently saved)
if metadata_file_name==None or metadata_file_name.lower()=="default" or metadata_file_name=="":
    
    # import target files in the metadata_directory
    metadata_files = listdirNHF(metadata_directory,
                                target=default_metadata_file_target,
                                exclude=default_metadata_file_exclude)
    
    # get the default metadata file
    metadata_file_name = default_file_name(file_list=metadata_files,
                                           from_file_name=metadata_from_file_name,
                                           directory_path=metadata_directory,
                                           separator=metadata_default_separator,
                                           date_position=metadata_default_date_position,
                                           date_format=metadata_default_date_format,
                                           reverse=metadata_default_reverse)
    
    print(f"using {metadata_file_name} as default metadata file")


# open the metadata file
metadata_df_i = pd.read_csv(os.path.join(metadata_directory, metadata_file_name))

# copy metadata_df
metadata_df = metadata_df_i.copy()

# # display the metadata dataframe
# metadata_df



using 20260114_ACID_metadata_part_3.csv as default metadata file


#### Select train data set - NOTE: the metadata_df is updated into a metadata_df which does not contain the test data

Run the following cell.

Don't modify the following cell.

In [5]:
# Select only the train set and update metadata_df
metadata_df = metadata_df[metadata_df[is_train_column] == train_val]

# assert proper selection of train set
assert metadata_df.shape[0] > 0, "No rows in metadata_df belong to the train set."
assert all(metadata_df[is_train_column] == train_val), "Not all rows in metadata_df belong to the train set."

# Display the metadata dataframe
metadata_df


,raw_file_name,scene_name,processing_date_yymmdd,ome_tif_file_name,location,microscope,objective,experiment,condition1,infectious_organism,...,top_percentile_fraction-1,top_percentile_fraction-2,top_percentile_fraction-3,top_percentile_fraction-4,mean_over_std-0,mean_over_std-1,mean_over_std-2,mean_over_std-3,mean_over_std-4,flag
0,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A1,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A1...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020022,0.020022,0.022416,0.020002,2.585605,1.195264,1.445345,27.645255,17.653226,0
1,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A2,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020003,0.020005,0.021515,0.020010,2.159470,1.592156,1.751387,25.278685,15.561817,0
2,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A5,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A5...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020011,0.020014,0.022187,0.020006,2.179615,1.817711,1.600485,27.096500,15.971792,0
3,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A6,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020000,0.020003,0.021979,0.020007,2.243834,1.904663,1.444320,27.040939,15.403914,0
4,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,B7,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B7...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020019,0.020011,0.021446,0.020018,2.278106,2.020556,1.607375,27.431643,15.968074,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
818,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,F3,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_F3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020432,0.020003,0.021571,0.020041,1.960308,19.792109,1.524763,27.921094,13.773708,0
819,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G2,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020045,0.020019,0.021691,0.020070,1.827073,8.612130,1.740165,27.868546,15.414203,0
820,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G4,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G4...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020598,0.020002,0.021369,0.020025,1.789514,14.308079,1.693619,27.543277,14.619977,0
821,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G6,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020006,0.020002,0.021918,0.020114,2.301495,5.008123,1.532941,18.554031,17.062070,0


#### Select non flagged fields of view - NOTE: the metadata_df is updated into a metadata_df which does not contain flagged data

Run the following cell.

Don't modify the following cell.

In [6]:
# Select only non-flagged rows and update metadata_df
metadata_df = metadata_df[metadata_df[flag_column] != flag_value]

# assert proper selection of train set
assert metadata_df.shape[0] > 0, "No rows in metadata_df belong haven't been flagged."
assert all(metadata_df[flag_column] != flag_value), "Some rows in metadata_df are flagged after selection."

# Display the metadata dataframe
metadata_df


,raw_file_name,scene_name,processing_date_yymmdd,ome_tif_file_name,location,microscope,objective,experiment,condition1,infectious_organism,...,top_percentile_fraction-1,top_percentile_fraction-2,top_percentile_fraction-3,top_percentile_fraction-4,mean_over_std-0,mean_over_std-1,mean_over_std-2,mean_over_std-3,mean_over_std-4,flag
0,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A1,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A1...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020022,0.020022,0.022416,0.020002,2.585605,1.195264,1.445345,27.645255,17.653226,0
1,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A2,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020003,0.020005,0.021515,0.020010,2.159470,1.592156,1.751387,25.278685,15.561817,0
2,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A5,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A5...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020011,0.020014,0.022187,0.020006,2.179615,1.817711,1.600485,27.096500,15.971792,0
3,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A6,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020000,0.020003,0.021979,0.020007,2.243834,1.904663,1.444320,27.040939,15.403914,0
4,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,B7,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B7...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,0.020019,0.020011,0.021446,0.020018,2.278106,2.020556,1.607375,27.431643,15.968074,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
818,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,F3,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_F3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020432,0.020003,0.021571,0.020041,1.960308,19.792109,1.524763,27.921094,13.773708,0
819,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G2,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020045,0.020019,0.021691,0.020070,1.827073,8.612130,1.740165,27.868546,15.414203,0
820,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G4,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G4...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020598,0.020002,0.021369,0.020025,1.789514,14.308079,1.693619,27.543277,14.619977,0
821,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G6,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,0.020006,0.020002,0.021918,0.020114,2.301495,5.008123,1.532941,18.554031,17.062070,0


In [7]:
metadata_df.columns

Index(['raw_file_name', 'scene_name', 'processing_date_yymmdd',
       'ome_tif_file_name', 'location', 'microscope', 'objective',
       'experiment', 'condition1', 'infectious_organism', 'condition_2',
       'imaging_hours', 'well', 'channel_0', 'channel_1', 'channel_2',
       'channel_3', 'channel_4', 'physical_size_unit_x',
       'physical_size_unit_y', 'dtype', 'size_t', 'size_c', 'size_z', 'size_y',
       'physical_size_y', 'size_x', 'physical_size_x', 'dims_order',
       'int_well', 'treatment', 'is_train', 'plls-0', 'plls-1', 'plls-2',
       'plls-3', 'plls-4', 'bottom_percentile_fraction-0',
       'bottom_percentile_fraction-1', 'bottom_percentile_fraction-2',
       'bottom_percentile_fraction-3', 'bottom_percentile_fraction-4',
       'top_percentile_fraction-0', 'top_percentile_fraction-1',
       'top_percentile_fraction-2', 'top_percentile_fraction-3',
       'top_percentile_fraction-4', 'mean_over_std-0', 'mean_over_std-1',
       'mean_over_std-2', 'mean_over_std

### Get all fields of view belonging to a specific position in the well (aka a scene)

In [8]:
scene_fov = metadata_df.groupby('scene_name')['ome_tif_file_name'].apply(np.array)



In [37]:
# get the number of channels in the images indicated in the metadata dataframe
fov_shape, num_channels, shape_of_channels = get_fov_ch_shape(metadata_df,
                                                              fov_directory,
                                                              fov_clm=fov_column_name,
                                                              channel_axis=channel_axis,
                                                              null_value=null_value)

for scene in scene_fov:
    container_arr_shape = list(fov_shape)
    container_arr_shape.append(scene.shape[0])
    container_arr = np.zeros(container_arr_shape)
    print(container_arr.shape)
    for fov_i, fov_name in enumerate(scene):
        field_of_view = tifffile.imread(os.path.join(fov_directory, fov_name))
        container_arr[...,fov_i] = field_of_view

field of view shape: (5, 1024, 1024)
number of channels found: 5
shape of channels: (1024, 1024)
(5, 1024, 1024, 15)
(5, 1024, 1024, 21)
(5, 1024, 1024, 11)
(5, 1024, 1024, 15)
(5, 1024, 1024, 17)
(5, 1024, 1024, 21)
(5, 1024, 1024, 14)
(5, 1024, 1024, 17)
(5, 1024, 1024, 16)
(5, 1024, 1024, 19)
(5, 1024, 1024, 16)
(5, 1024, 1024, 18)
(5, 1024, 1024, 19)
(5, 1024, 1024, 16)
(5, 1024, 1024, 17)
(5, 1024, 1024, 16)
(5, 1024, 1024, 19)
(5, 1024, 1024, 13)
(5, 1024, 1024, 17)
(5, 1024, 1024, 17)
(5, 1024, 1024, 22)
(5, 1024, 1024, 15)
(5, 1024, 1024, 21)
(5, 1024, 1024, 17)
(5, 1024, 1024, 15)
(5, 1024, 1024, 17)
(5, 1024, 1024, 18)
(5, 1024, 1024, 16)
(5, 1024, 1024, 13)
(5, 1024, 1024, 15)
(5, 1024, 1024, 18)
(5, 1024, 1024, 15)
(5, 1024, 1024, 16)
(5, 1024, 1024, 16)
(5, 1024, 1024, 18)
(5, 1024, 1024, 15)
(5, 1024, 1024, 19)
(5, 1024, 1024, 17)
(5, 1024, 1024, 14)
(5, 1024, 1024, 14)
(5, 1024, 1024, 20)
(5, 1024, 1024, 16)
(5, 1024, 1024, 17)
(5, 1024, 1024, 19)
(5, 1024, 1024, 15)
(5,

KeyboardInterrupt: 

### Get all fields of view belonging to a specific well

In [10]:
well_fov = metadata_df.groupby('well')['ome_tif_file_name'].apply(np.array)


In [11]:
for w in well_fov:
    print(w.shape)

(104,)
(102,)
(104,)
(105,)
(98,)
(104,)
(97,)
(103,)


#### Add flag column to the dataset and save the results

Run the following cell.

Don't modify the following cell.

In [12]:
# # concatenate QC columns names
# qc_column_names = plls_column_names + bottom_perc_fract_column_names + top_perc_fract_column_names + mean_over_std_column_names

# # concatenate QC lowpass thresholds
# qc_lowpass_thresholds = plls_lowpass_thresholds +  bot_perc_lowpass_thresholds + top_perc_lowpass_thresholds + mean_over_std_lowpass_thresholds

# # concatenate QC highpass thresholds
# qc_highpass_thresholds = plls_highpass_thresholds + bot_perc_highpass_thresholds + top_perc_highpass_thresholds + mean_over_std_highpass_thresholds

# # add flag column
# flagged_metadata_df = add_flag_column(df=metadata_df,
#                                       lowpass_thres=qc_lowpass_thresholds,
#                                       highpass_thres=qc_highpass_thresholds,
#                                       cols=qc_column_names,
#                                       flag_col="flag",
#                                       flag_value=1,
#                                       ok_value=0)

# # save the dataframe
# flagged_metadata_saving_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{metadata_savingword}{save_file_name_separator}{metadata_file_suffix}"
# flagged_metadata_df.to_csv(os.path.join(output_directory, flagged_metadata_saving_name), index=save_csv_index)


### Save hyperparameters

Run the following cell.

Don't modify the following cell.

In [13]:
# # # collect hyperparameters in a dictionary

# hyperparameter_dict = {

# 'metadata_directory':metadata_directory,
# 'fov_directory':fov_directory,
# 'output_directory':output_directory,
# 'metadata_file_name':metadata_file_name,
# 'is_train_column':is_train_column,
# 'train_val':train_val,
# 'default_metadata_file_target':default_metadata_file_target,
# 'default_metadata_file_exclude':default_metadata_file_exclude,
# 'metadata_from_file_name':metadata_from_file_name,
# 'metadata_default_separator':metadata_default_separator,
# 'metadata_default_date_position':metadata_default_date_position,
# 'metadata_default_date_format':metadata_default_date_format,
# 'metadata_default_reverse':metadata_default_reverse,
# 'channel_axis':channel_axis,
# 'fov_column_name':fov_column_name,
# 'null_value':null_value,
# 'plls_column_name':plls_column_name,
# 'low_percentile':low_percentile,
# 'high_percentile':high_percentile,
# 'bottom_perc_fract_column_name':bottom_perc_fract_column_name,
# 'top_perc_fract_column_name':top_perc_fract_column_name,
# 'mean_over_std_column_name':mean_over_std_column_name,
# 'ch_measurement_separator':ch_measurement_separator,
# 'save_file_name_separator':save_file_name_separator,
# 'project_name':project_name,
# 'save_csv_index':save_csv_index,
# 'metadata_date_format':metadata_date_format,
# 'metadata_savingword':metadata_savingword,
# 'metadata_file_suffix':metadata_file_suffix,
# 'hyperparameters_date_format':hyperparameters_date_format,
# 'hyperparameters_savingword':hyperparameters_savingword,
# 'hyperparameters_file_suffix':hyperparameters_file_suffix,
# 'secondary_output_directory':secondary_output_directory,
# 'exist_ok':exist_ok,
# 'plls_lowpass_thresholds':plls_lowpass_thresholds,
# 'bot_perc_lowpass_thresholds':bot_perc_lowpass_thresholds,
# 'top_perc_lowpass_thresholds':top_perc_lowpass_thresholds,
# 'mean_over_std_lowpass_thresholds':mean_over_std_lowpass_thresholds,
# 'plls_highpass_thresholds':plls_highpass_thresholds,
# 'bot_perc_highpass_thresholds':bot_perc_highpass_thresholds,
# 'top_perc_highpass_thresholds':top_perc_highpass_thresholds,
# 'mean_over_std_highpass_thresholds':mean_over_std_highpass_thresholds
# }

# # transform the hyperparameter_dict in a pandas series
# hyperparameter_series = pd.Series(hyperparameter_dict)

# # save hyperparamters
# hyperparameter_saving_name = f"{datetime.datetime.now().strftime(hyperparameters_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{hyperparameters_savingword}{save_file_name_separator}{hyperparameters_file_suffix}"
# hyperparameter_series.to_csv(os.path.join(secondary_output_directory,hyperparameter_saving_name), index=save_csv_index)

